# 2026-10 — Καθάρισμα Neon μετά την επισκευή των fundamentals

**Τι κάνει:** σβήνει τους προσωρινούς πίνακες backup που δημιουργήθηκαν στη Neon κατά την επισκευή (7–13/10/2026).

**Προϋπόθεση:** η βάση να είναι υγιής (κελί 2 → ✅). Αν δεν είναι, το κελί 4 **αρνείται** να σβήσει οτιδήποτε.

**Τι ΔΕΝ αγγίζει:**
- τον πίνακα `fundamentals` (μόνο διαβάζει),
- τα τοπικά backups σε parquet (`backup_20261007/`) — μένουν στον υπολογιστή, δεν πιάνουν χώρο στη Neon.

Τρέχει από τον φάκελο `shared data layer` (εκεί είναι το `db.py` και το `fundamentals.py`).

In [1]:
# ── 1. Setup ─────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path
from sqlalchemy import text

import fundamentals as F
from db import get_engine

eng = get_engine()
print("fundamentals από:", F.__file__)
print("Stage 0 φορτωμένο:", hasattr(F, "OWN_PERIOD_MIN_SHARE"))   # πρέπει True

fundamentals από: C:\Users\georg\Google Drive\Python Finance\shared data layer\fundamentals.py
Stage 0 φορτωμένο: True


## 2. Έλεγχος υγείας της βάσης

Οι ίδιοι έλεγχοι με το `integrity_report()` του `single_asset_analyzer`:
- **(β)** ίδια περίοδος με δύο fiscal_year → πρέπει **0**
- **(γ)** ίδια τιμή + ημερομηνία σε 3+ περιόδους → πρέπει **0**
- **(α)** lag > 300 μέρες → πληροφοριακό (σημείο αναφοράς: 10-Q ~288, 10-K ~85)

In [2]:
# ── 2. Integrity ─────────────────────────────────────────────────────────
SKIP = ["PARA", "XOM"]   # δύο CIK στο ίδιο ticker — γνωστό, εκτός επισκευής
P = {"canon": list(F.XBRL_TAG_MAP), "skip": SKIP}
BASE = """FROM fundamentals f
          JOIN metric_lookup m     ON m.id  = f.metric_id
          JOIN form_type_lookup ft ON ft.id = f.form_type_id
          WHERE m.name = ANY(:canon) AND NOT (f.ticker = ANY(:skip))"""

with eng.connect() as c:
    late = pd.read_sql(text(f"""
        SELECT ft.name AS form, COUNT(*) AS rows, COUNT(DISTINCT f.ticker) AS tickers
        {BASE} AND f.filed_date - f.period_end > 300
        GROUP BY 1 ORDER BY 2 DESC"""), c, params=P)
    dup = c.execute(text(f"""
        SELECT COUNT(*), COUNT(DISTINCT ticker) FROM (
            SELECT f.ticker, f.metric_id, f.form_type_id, f.period_end
            {BASE} AND f.period_start IS NOT NULL
            GROUP BY 1, 2, 3, 4 HAVING COUNT(DISTINCT f.fiscal_year) > 1) x"""), P).fetchone()
    copied = c.execute(text(f"""
        SELECT COUNT(*), COUNT(DISTINCT ticker) FROM (
            SELECT f.ticker, f.metric_id, f.period_end, f.value
            {BASE} AND ft.name IN ('10-Q', '10-K')
            GROUP BY 1, 2, 3, 4
            HAVING COUNT(DISTINCT (f.fiscal_year, f.fiscal_period)) >= 3) x"""), P).fetchone()
    last_ingest = c.execute(text("SELECT MAX(inserted_at) FROM fundamentals")).scalar()

DB_OK = dup[0] == 0 and copied[0] == 0
print("(α) lag > 300 μέρες:"); print(late.to_string(index=False) if len(late) else "    0")
print(f"(β) ίδια περίοδος, δύο fiscal_year : {dup[0]:,} σε {dup[1]} tickers")
print(f"(γ) ίδια τιμή σε 3+ περιόδους      : {copied[0]:,} σε {copied[1]} tickers")
print(f"τελευταία εγγραφή στη βάση: {last_ingest}")
print("✅ OK" if DB_OK else "❌ ΠΡΟΣΟΧΗ — ΜΗΝ προχωρήσεις στο καθάρισμα")

(α) lag > 300 μέρες:
  form  rows  tickers
   8-K   876       29
10-Q/A   699       12
10-K/A   694       24
  10-Q   288        3
  10-K    85        2
 8-K/A    35        1
(β) ίδια περίοδος, δύο fiscal_year : 0 σε 0 tickers
(γ) ίδια τιμή σε 3+ περιόδους      : 0 σε 0 tickers
τελευταία εγγραφή στη βάση: 2026-10-14 05:58:43.485565+00:00
✅ OK


## 3. Τι υπάρχει προς καθάρισμα

Ψάχνει πίνακες με όνομα `fundamentals_bad_%` ή `fundamentals_backup_%`. Τυπώνει μέγεθος και γραμμές — τίποτα δεν σβήνεται εδώ.

In [3]:
# ── 3. Λίστα πινάκων backup ──────────────────────────────────────────────
with eng.connect() as c:
    tables = pd.read_sql(text("""
        SELECT tablename,
               pg_size_pretty(pg_total_relation_size(quote_ident(tablename))) AS size
        FROM pg_tables
        WHERE schemaname = 'public'
          AND (tablename LIKE 'fundamentals\\_bad\\_%' OR tablename LIKE 'fundamentals\\_backup\\_%')
        ORDER BY 1"""), c)
    tables["rows"] = [c.execute(text(f'SELECT COUNT(*) FROM "{t}"')).scalar() for t in tables["tablename"]]
    size_before = c.execute(text("SELECT pg_size_pretty(pg_database_size(current_database()))")).scalar()

print(tables.to_string(index=False) if len(tables) else "Κανένας πίνακας backup — δεν υπάρχει κάτι να σβηστεί.")
print(f"\nμέγεθος βάσης τώρα: {size_before}")

                   tablename  size  rows
fundamentals_backup_20261007 80 kB   477

μέγεθος βάσης τώρα: 390 MB


## 4. Διαγραφή

Δύο δικλείδες:
1. αν το κελί 2 δεν έβγαλε ✅, **σταματά**,
2. σβήνει **μόνο** τους πίνακες της λίστας του κελιού 3 — ποτέ τον `fundamentals`.

Άλλαξε `CONFIRM = True` για να τρέξει.

In [7]:
# ── 4. DROP ──────────────────────────────────────────────────────────────
CONFIRM = True   # ← άλλαξέ το σε True αφού δεις τα κελιά 2 και 3

assert DB_OK, "Η βάση δεν είναι ✅ — δεν σβήνουμε backups ενώ υπάρχει πρόβλημα."
to_drop = [t for t in tables["tablename"] if t.startswith(("fundamentals_bad_", "fundamentals_backup_"))]
assert "fundamentals" not in to_drop

if not to_drop:
    print("Τίποτα να σβηστεί.")
elif not CONFIRM:
    print(f"Θα σβηστούν: {to_drop}\n→ βάλε CONFIRM = True και ξανατρέξε το κελί.")
else:
    with eng.begin() as c:
        for t in to_drop:
            c.execute(text(f'DROP TABLE IF EXISTS "{t}"'))
            print(f"🗑️  {t}")
    print("✅ έγινε")

🗑️  fundamentals_backup_20261007
✅ έγινε


## 5. Μετά

Επιβεβαίωση ότι οι πίνακες έφυγαν και ότι ο `fundamentals` είναι ανέγγιχτος.

Το μέγεθος της βάσης μπορεί να μην πέσει αμέσως — η Neon/Postgres επιστρέφει τον χώρο σταδιακά. Ο πίνακας backup ήταν μικρός (~370 γραμμές), οπότε η διαφορά είναι ούτως ή άλλως αμελητέα.

In [8]:
# ── 5. Έλεγχος ───────────────────────────────────────────────────────────
with eng.connect() as c:
    left = c.execute(text("""
        SELECT COUNT(*) FROM pg_tables WHERE schemaname = 'public'
          AND (tablename LIKE 'fundamentals\\_bad\\_%' OR tablename LIKE 'fundamentals\\_backup\\_%')""")).scalar()
    n_fund = c.execute(text("SELECT COUNT(*) FROM fundamentals")).scalar()
    size_after = c.execute(text("SELECT pg_size_pretty(pg_database_size(current_database()))")).scalar()

print(f"πίνακες backup που έμειναν: {left}   (πρέπει 0)")
print(f"γραμμές στο fundamentals:   {n_fund:,}")
print(f"μέγεθος βάσης: {size_before} → {size_after}")

πίνακες backup που έμειναν: 0   (πρέπει 0)
γραμμές στο fundamentals:   1,319,274
μέγεθος βάσης: 390 MB → 390 MB


## 6. Τοπικά backups (πληροφοριακό)

Τα parquet της επισκευής μένουν στον υπολογιστή. Είναι στο `.gitignore`, δεν ανεβαίνουν στο GitHub.
Πρόταση: κράτα τα μερικούς μήνες. Αν ποτέ χρειαστεί επαναφορά ενός ticker: `repair_20261007.restore_ticker(ticker, eng)`.

In [9]:
# ── 6. Τοπικά parquet ────────────────────────────────────────────────────
d = Path("backup_20261007")
files = sorted(d.glob("*.parquet")) if d.exists() else []
mb = sum(f.stat().st_size for f in files) / 2**20
print(f"{len(files)} αρχεία, {mb:.1f} MB στο {d.resolve()}")

1194 αρχεία, 12.7 MB στο C:\Users\georg\Google Drive\Python Finance\shared data layer\backup_20261007
